In [1]:
import pandas as pd
import numpy as np

In [3]:
simple_prompt_fine_tune_full_df = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_imageclef2026-rag_20260408_084221.csv")
simple_prompt_fine_tune_full_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI sagittal T1 weighted image showing t...
1,ImageCLEFmedical_Caption_2026_valid_1,Axial magnetic resonance image with contrast s...
2,ImageCLEFmedical_Caption_2026_valid_2,Lumbar spine MRI shows high-intensity signal o...
3,ImageCLEFmedical_Caption_2026_valid_3,Preoperative radiograph. The yellow arrow show...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvis radiograph showing bilateral displaced ...


In [4]:
df_ref = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_simple_prompt_20260310_113904/simple_prompt_config_imageclef2026-rag_simple_prompt_run_20260310_113904.csv")
df_ref.shape

(19240, 2)

In [5]:
simple_prompt_fine_tune_full_df.shape

(19232, 2)

In [8]:
images_missing = set(df_ref["ID"]) - set(simple_prompt_fine_tune_full_df["ID"])
print(f"Number of missing images: {len(images_missing)}")
print("Missing image IDs: ", images_missing)

Number of missing images: 8
Missing image IDs:  {'ImageCLEFmedical_Caption_2026_valid_12811', 'ImageCLEFmedical_Caption_2026_valid_12807', 'ImageCLEFmedical_Caption_2026_valid_12808', 'ImageCLEFmedical_Caption_2026_valid_12809', 'ImageCLEFmedical_Caption_2026_valid_12813', 'ImageCLEFmedical_Caption_2026_valid_12814', 'ImageCLEFmedical_Caption_2026_valid_12810', 'ImageCLEFmedical_Caption_2026_valid_12812'}


In [9]:
# there are any duplicated IDs in the new df?
duplicated_ids = simple_prompt_fine_tune_full_df[simple_prompt_fine_tune_full_df.duplicated(subset=["ID"], keep=False)]
print(f"Number of duplicated IDs: {duplicated_ids.shape[0]}")
if not duplicated_ids.empty:
    print("Duplicated IDs: ", duplicated_ids["ID"].tolist())

Number of duplicated IDs: 0


In [10]:
len(simple_prompt_fine_tune_full_df["ID"].unique())

19232

## rodar para IDs faltantes

In [16]:
# ids faltantes = images_missing
from anyio import Path
import json

base = Path("/home/ia368/projetos/imageclef2026-rag")

In [17]:
print("IDs selecionados:", len(images_missing))
print(images_missing)

dataset_full = base / "artifacts/datasets/imageclef2026_valid_dataset.json"

with open(dataset_full, "r", encoding="utf-8") as f:
    data = json.load(f)

subset = [x for x in data if x["image_id"] in set(images_missing)]
found_ids = {x["image_id"] for x in subset}
not_found = [x for x in images_missing if x not in found_ids]

print("Encontrados no dataset:", len(subset))
print("Nao encontrados:", not_found)

subset_path = base / "artifacts/results/simple_prompt_fine_tune_full_20260408_084221/imageclef2026_valid_subset_8_ids.json"
with open(subset_path, "w", encoding="utf-8") as f:
    json.dump(subset, f, ensure_ascii=False, indent=2)

print("Subset salvo em:", subset_path)

IDs selecionados: 8
{'ImageCLEFmedical_Caption_2026_valid_12811', 'ImageCLEFmedical_Caption_2026_valid_12807', 'ImageCLEFmedical_Caption_2026_valid_12808', 'ImageCLEFmedical_Caption_2026_valid_12809', 'ImageCLEFmedical_Caption_2026_valid_12813', 'ImageCLEFmedical_Caption_2026_valid_12814', 'ImageCLEFmedical_Caption_2026_valid_12810', 'ImageCLEFmedical_Caption_2026_valid_12812'}
Encontrados no dataset: 8
Nao encontrados: []
Subset salvo em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/imageclef2026_valid_subset_8_ids.json


In [18]:
import yaml
from copy import deepcopy

config_src = base / "configs/runs/simple_prompt_fine_tuned_full.yaml"
config_tmp = base / "artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_8_ids.yaml"

with open(config_src, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg2 = deepcopy(cfg)
cfg2["dataset"] = str(subset_path)

with open(config_tmp, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg2, f, sort_keys=False, allow_unicode=True)

print("Config temporaria salva em:", config_tmp)

Config temporaria salva em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_8_ids.yaml


In [20]:
csv_main = base / "/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_imageclef2026-rag_20260408_084221.csv"
cmd = f'python {base/"run_med_gemma.py"} --config {config_tmp} --resume_csv {csv_main}'
print(cmd)

python /home/ia368/projetos/imageclef2026-rag/run_med_gemma.py --config /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_8_ids.yaml --resume_csv /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_imageclef2026-rag_20260408_084221.csv


## alterar ordem os IDs no arquivo csv final

In [38]:
simple_prompt_fine_tune_full_df = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_imageclef2026-rag_20260408_084221.csv")
simple_prompt_fine_tune_full_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI sagittal T1 weighted image showing t...
1,ImageCLEFmedical_Caption_2026_valid_1,Axial magnetic resonance image with contrast s...
2,ImageCLEFmedical_Caption_2026_valid_2,Lumbar spine MRI shows high-intensity signal o...
3,ImageCLEFmedical_Caption_2026_valid_3,Preoperative radiograph. The yellow arrow show...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvis radiograph showing bilateral displaced ...


In [39]:
simple_prompt_fine_tune_full_df['id_num'] = (
    simple_prompt_fine_tune_full_df['ID'].str.split('_').str[-1].astype(int)
)
simple_prompt_fine_tune_full_df.head()

,ID,Caption,id_num
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI sagittal T1 weighted image showing t...,0
1,ImageCLEFmedical_Caption_2026_valid_1,Axial magnetic resonance image with contrast s...,1
2,ImageCLEFmedical_Caption_2026_valid_2,Lumbar spine MRI shows high-intensity signal o...,2
3,ImageCLEFmedical_Caption_2026_valid_3,Preoperative radiograph. The yellow arrow show...,3
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvis radiograph showing bilateral displaced ...,4


In [40]:
simple_prompt_fine_tune_full_df = simple_prompt_fine_tune_full_df.sort_values(by="id_num").reset_index()
simple_prompt_fine_tune_full_df.head()

,index,ID,Caption,id_num
0,0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI sagittal T1 weighted image showing t...,0
1,1,ImageCLEFmedical_Caption_2026_valid_1,Axial magnetic resonance image with contrast s...,1
2,2,ImageCLEFmedical_Caption_2026_valid_2,Lumbar spine MRI shows high-intensity signal o...,2
3,3,ImageCLEFmedical_Caption_2026_valid_3,Preoperative radiograph. The yellow arrow show...,3
4,4,ImageCLEFmedical_Caption_2026_valid_4,Pelvis radiograph showing bilateral displaced ...,4


In [41]:
simple_prompt_fine_tune_full_df.tail(10)

,index,ID,Caption,id_num
19230,19222,ImageCLEFmedical_Caption_2026_valid_19243,Anterior chamber depth was measured with anter...,19243
19231,19223,ImageCLEFmedical_Caption_2026_valid_19244,AS-OCT image of a patient with diabetic macula...,19244
19232,19224,ImageCLEFmedical_Caption_2026_valid_19245,Macular swelling.,19245
19233,19225,ImageCLEFmedical_Caption_2026_valid_19246,A sample image from the DICOM dataset for the ...,19246
19234,19226,ImageCLEFmedical_Caption_2026_valid_19247,Optical coherence tomography scan showing a de...,19247
19235,19227,ImageCLEFmedical_Caption_2026_valid_19248,Spectral-domain optical coherence tomography i...,19248
19236,19228,ImageCLEFmedical_Caption_2026_valid_19249,Retinal OCT. Red arrow: intravitreal fluid; bl...,19249
19237,19229,ImageCLEFmedical_Caption_2026_valid_19250,Optical coherence tomography scan of the right...,19250
19238,19230,ImageCLEFmedical_Caption_2026_valid_19251,"Anterior segment OCT showing a large, shallow ...",19251
19239,19231,ImageCLEFmedical_Caption_2026_valid_19252,Anterior segment optical coherence tomography ...,19252


In [42]:
simple_prompt_fine_tune_full_df.shape

(19240, 4)

In [43]:
simple_prompt_fine_tune_full_df.drop(columns=["index", "id_num"], inplace=True)
simple_prompt_fine_tune_full_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI sagittal T1 weighted image showing t...
1,ImageCLEFmedical_Caption_2026_valid_1,Axial magnetic resonance image with contrast s...
2,ImageCLEFmedical_Caption_2026_valid_2,Lumbar spine MRI shows high-intensity signal o...
3,ImageCLEFmedical_Caption_2026_valid_3,Preoperative radiograph. The yellow arrow show...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvis radiograph showing bilateral displaced ...


In [44]:
simple_prompt_fine_tune_full_df.to_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_fine_tune_full_20260408_084221/simple_prompt_fine_tuned_full_imageclef2026-rag_20260408_084221_sorted.csv", index=False)